# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/haideriqbal499/Flyrank_Ml_internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

**Lane 2 — Refresh / Content Opportunity Scoring.**
This notebook writes the data contract in plain words, proves three facts on a mid-panel month
(`month=2026-03`), builds five honest features, and springs the label-leakage trap once.

> Skills: `writing-data-contracts` + `flyrank/flyrank-data`.

## 0. Setup — DuckDB + Hugging Face token

Request access on [`FlyRank/internship-warehouse`](https://huggingface.co/datasets/FlyRank/internship-warehouse),
then use a **READ** token via env var / Colab Secret `HF_TOKEN` — never paste a token into a committed cell.

In [ ]:
import os
import sys
import subprocess

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "duckdb", "huggingface_hub", "pandas", "scikit-learn"], check=True)

import getpass
import duckdb
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass("Paste your Hugging Face READ token (hf_...): ")
assert HF_TOKEN and HF_TOKEN.startswith("hf_"), "Need a Hugging Face READ token starting with hf_"

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
# Mid-panel month only — never develop labels on the _sample (June 2026 = sealed test month).
FACT_MAR = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"
DIM_CONTENT = f"read_parquet('{REL}/dim_content.parquet')"
DIM_CLIENTS = f"read_parquet('{REL}/dim_clients.parquet')"

print("Connected. Working month partition: 2026-03")

## 1. Unit of analysis + time window

### Contract answers (plain words) — part A

| # | Question | My answer |
|---|---|---|
| 1 | **What one row means (for my lane)** | For the warehouse daily fact: one row = one `report_date` × one `client_hash_id` × one `content_hash_id`. For the Lane 2 model frame I build below: one row = **one content item** (page), with features rolled up from a past window. |
| 2 | **Which table(s)** | `fact_content_daily_performance` (partition `month=2026-03`) joined to `dim_content` for page metadata. `dim_clients` only for history-coverage context — not as model features. |
| 3 | **Which time window** | Iteration window: **March 2026** only. Inside that month I split days 1–15 (feature window) vs days 16–31 (outcome window). Decision moment = end of day 15. Later capstone work will use a true next-month label and keep June 2026 sealed. |

In [ ]:
# Smoke check: partition exists and has dates inside March 2026
smoke = con.sql(f"""
    SELECT COUNT(*) AS n_rows,
           MIN(report_date) AS min_d,
           MAX(report_date) AS max_d
    FROM {FACT_MAR}
""").df()
print(smoke.to_string(index=False))
assert smoke.loc[0, "min_d"].month == 3 and smoke.loc[0, "max_d"].month == 3
print("Smoke OK: March 2026 partition readable.")

## 2. Fields: feature / label / context / excluded

### Contract answers (plain words) — part B

| # | Question | My answer |
|---|---|---|
| 4 | **What I predict / rank (label or proxy)** | **Proxy label for this notebook:** `needs_refresh_review` = 1 when impressions in days 16–31 fall below 80% of impressions in days 1–15 (and days 1–15 had enough demand). Real Lane 2 deliverable is a **ranked review queue**; this binary proxy lets me score honesty of features. Preferred later label: observed decline in a *later* calendar month from prior-window features. |
| 5 | **One thing I deliberately exclude** | **`fact_content_query_90d`** for this mid-panel iteration — its fixed 90-day window sits at the *end* of the panel, not inside March 2026. Using it here would mix the wrong calendar into features (and leak if the label ever touches those final months). Also excluded as features: `client_hash_id` / `content_hash_id` (context only), and any GA4 metric on rows where `ga4_data_available IS NOT TRUE`. |

### Field buckets (fields I touch)

| Bucket | Fields |
|---|---|
| **Feature** | `imp_first15`, `clk_first15`, `pos_first15`, `days_with_imp_first15`, `ctr_first15` |
| **Label / proxy** | `imp_last16`, `needs_refresh_review` (and anything derived from days 16–31) |
| **Context** | `client_hash_id`, `content_hash_id`, `report_date`, `ga4_data_available` |
| **Excluded** | `fact_content_query_90d` (wrong window); GA4 zeros when flag is not TRUE; product decision flags (not in release) |

In [ ]:
# Show which availability flags appear in March (three-valued: TRUE / FALSE / NULL)
flags = con.sql(f"""
    SELECT
        ga4_data_available,
        COUNT(*) AS n_rows
    FROM {FACT_MAR}
    GROUP BY 1
    ORDER BY 1 NULLS LAST
""").df()
print("ga4_data_available value counts in month=2026-03:")
print(flags.to_string(index=False))

## 3. Verify it with queries + five features + the trap

Exactly **three** verification queries on `month=2026-03`, then the feature frame and leakage lesson.

### Query 1 — Grain

Claim: one daily-fact row is uniquely identified by `(report_date, client_hash_id, content_hash_id)`.
Check: duplicate groups should return **zero** rows.

In [ ]:
grain = con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS c
    FROM {FACT_MAR}
    GROUP BY 1, 2, 3
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()
print("Duplicate grain groups (expect empty):", len(grain))
print(grain)
assert len(grain) == 0, "Grain broken — unexpected duplicate keys"
print("Grain holds for month=2026-03.")

### Query 2 — Slice size and date span

Claim: this partition is March 2026 and has a large daily×page row count.

In [ ]:
span = con.sql(f"""
    SELECT
        COUNT(*) AS n_rows,
        COUNT(DISTINCT client_hash_id) AS n_clients,
        COUNT(DISTINCT content_hash_id) AS n_content,
        MIN(report_date) AS min_report_date,
        MAX(report_date) AS max_report_date
    FROM {FACT_MAR}
""").df()
print(span.to_string(index=False))
assert str(span.loc[0, "min_report_date"])[:7] == "2026-03"
assert str(span.loc[0, "max_report_date"])[:7] == "2026-03"
print("Date span confirmed inside March 2026.")

### Query 3 — Availability (`IS TRUE`)

Claim: GA4 columns are only trustworthy when `ga4_data_available IS TRUE`.
Filter with `IS TRUE` (not `= TRUE` / `NOT FALSE`) so NULL flags are not treated as usable.

In [ ]:
avail = con.sql(f"""
    SELECT
        COUNT(*) AS n_all,
        COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS n_ga4_true,
        COUNT(*) FILTER (WHERE ga4_data_available IS FALSE) AS n_ga4_false,
        COUNT(*) FILTER (WHERE ga4_data_available IS NULL) AS n_ga4_null,
        ROUND(100.0 * COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) / COUNT(*), 2) AS pct_ga4_true
    FROM {FACT_MAR}
""").df()
print(avail.to_string(index=False))
print("Rows surviving ga4_data_available IS TRUE:", int(avail.loc[0, "n_ga4_true"]))

### Five-feature frame (Lane 2)

Build one row per content item from March 2026. Features use **days 1–15 only**.
Label uses **days 16–31 only** (never as a feature).

| Feature | Knowable at the decision moment because… |
|---|---|
| `imp_first15` | Sum of GSC impressions on days 1–15 — fully observed before day-16 outcome starts. |
| `clk_first15` | Sum of GSC clicks on days 1–15 — same past window. |
| `pos_first15` | Average `gsc_avg_position` on days 1–15 where position is present — past only. |
| `days_with_imp_first15` | Count of days 1–15 with impressions > 0 — known at end of day 15. |
| `ctr_first15` | `clk_first15 / imp_first15` on the past window — no future clicks mixed in. |

In [ ]:
frame = con.sql(f"""
    WITH daily AS (
        SELECT
            client_hash_id,
            content_hash_id,
            report_date,
            gsc_impressions,
            gsc_clicks,
            NULLIF(gsc_avg_position, 0) AS pos_nonzero
        FROM {FACT_MAR}
    ),
    agg AS (
        SELECT
            client_hash_id,
            content_hash_id,
            SUM(CASE WHEN DAY(report_date) <= 15 THEN gsc_impressions ELSE 0 END) AS imp_first15,
            SUM(CASE WHEN DAY(report_date) <= 15 THEN gsc_clicks ELSE 0 END) AS clk_first15,
            AVG(CASE WHEN DAY(report_date) <= 15 THEN pos_nonzero END) AS pos_first15,
            SUM(CASE WHEN DAY(report_date) <= 15 AND gsc_impressions > 0 THEN 1 ELSE 0 END) AS days_with_imp_first15,
            SUM(CASE WHEN DAY(report_date) > 15 THEN gsc_impressions ELSE 0 END) AS imp_last16
        FROM daily
        GROUP BY 1, 2
    )
    SELECT
        *,
        CASE WHEN imp_first15 > 0 THEN clk_first15 * 1.0 / imp_first15 ELSE NULL END AS ctr_first15,
        CASE
            WHEN imp_first15 >= 50 AND imp_last16 < 0.8 * imp_first15 THEN 1
            ELSE 0
        END AS needs_refresh_review
    FROM agg
    WHERE imp_first15 >= 50   -- enough past demand to bother an editor
""").df()

feature_cols = ["imp_first15", "clk_first15", "pos_first15", "days_with_imp_first15", "ctr_first15"]
print(f"Feature frame rows (content items): {len(frame):,}")
print(f"Proxy positive rate: {frame['needs_refresh_review'].mean():.3f}")
print("Feature columns:", feature_cols)
frame[feature_cols + ["needs_refresh_review", "imp_last16"]].head()

### The trap — add one label-derived column, watch the score jump, then delete it

I deliberately add `imp_last16` (outcome-window impressions) as if it were a feature.
That is leakage: the model peeks at the future it is supposed to predict.
Then I remove it and keep the honest score.

In [ ]:
model_df = frame.dropna(subset=feature_cols + ["needs_refresh_review", "imp_last16"]).copy()
y = model_df["needs_refresh_review"].astype(int)

def quick_auc(cols, label):
    X = model_df[cols].fillna(0)
    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, test_size=0.25, random_state=42, stratify=y
    )
    clf = LogisticRegression(max_iter=1000)
    clf.fit(X_tr, y_tr)
    proba = clf.predict_proba(X_te)[:, 1]
    auc = roc_auc_score(y_te, proba)
    print(f"{label}: ROC AUC = {auc:.4f}  (features={cols})")
    return auc

print("--- honest features only ---")
auc_honest = quick_auc(feature_cols, "honest")

print("\n--- TRAP: add label-window column imp_last16 ---")
auc_leaky = quick_auc(feature_cols + ["imp_last16"], "leaky")

print("\n--- trap removed; keep honest number ---")
auc_final = quick_auc(feature_cols, "honest_again")

print(f"\nLeakage jump: {auc_leaky - auc_honest:+.4f} AUC when future impressions were included.")
print(f"Honest number to keep: ROC AUC = {auc_final:.4f}")
assert auc_leaky > auc_honest + 0.02, "Expected a clear jump from leakage — check label construction"
assert abs(auc_final - auc_honest) < 1e-9

## 4. Data limits

**Named limitation of this slice:** March 2026 is a single calendar month on an **unbalanced panel**.
Clients with short GSC history contribute fewer page-days, so row counts and decline rates are not
a simple random sample of “all web content.” A mid-month feature/label split is also only a
**teaching proxy** — seasonality inside March can look like “decline.” The real Lane 2 label should
come from a later month’s observed outcome, with June 2026 kept sealed as a test month.

Also: this notebook does not prove that refreshing a page recovers traffic — only that some pages
look worth reviewing first (decision-support).

In [ ]:
# Limitation receipt: history depth differs by client (unbalanced panel)
depth = con.sql(f"""
    SELECT
        COUNT(*) AS n_clients,
        COUNT(*) FILTER (WHERE gsc_data_start IS NOT NULL) AS n_with_gsc_start,
        MIN(gsc_data_start) AS earliest_gsc_start,
        MAX(gsc_data_start) AS latest_gsc_start
    FROM {DIM_CLIENTS}
""").df()
print("dim_clients history coverage (panel is unbalanced):")
print(depth.to_string(index=False))

## Self-check

Before you submit, confirm each line honestly:

- [x] Five plain-words contract answers (row, tables, window, label/proxy, exclude)
- [x] Exactly three verification queries with outputs (grain, span, `IS TRUE` availability)
- [x] Five-feature frame with an "available when?" line per feature
- [x] Deliberate leak shown (score jump) then removed; honest AUC kept
- [x] One named limitation of the slice (unbalanced panel + mid-month proxy)
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] Careful words: observed / directional / decision-support
- [ ] Committed under `work/notebooks/` — then submit the repo URL on the card